# 02 · Chunk + Ingest（写入 Chroma）

目标：
- 复用课件里推荐的 **MarkdownHeaderTextSplitter + RecursiveCharacterTextSplitter**
- 将 chunk 写入本地 **ChromaDB**（persistent）

输入：上一节解析得到的 Markdown（MinerU 或 PaddleOCR-VL 都可以）


In [2]:
from __future__ import annotations

import os
from pathlib import Path


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")


PROJECT_ROOT = resolve_project_root()

# 选择解析来源：mineru / paddleocr_vl
SOURCE = os.getenv("PARSE_SOURCE", "paddleocr_vl")
BASE_DIR = PROJECT_ROOT / "data/parsed/道通24年年报"

assert BASE_DIR.exists(), f"请先运行上一节解析：{BASE_DIR}"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE:", BASE_DIR)
print("SOURCE:", SOURCE)


PROJECT_ROOT: /Users/mengbai/Documents/AI-training/RAG_project
BASE: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报
SOURCE: paddleocr_vl


In [3]:
# 1) 读取 Markdown

from langchain_core.documents import Document


def load_markdown_documents(source: str) -> list[Document]:
    if source == "mineru":
        md_files = sorted((BASE_DIR / "mineru").glob("*.md"))
        assert md_files, "没找到 mineru 的 md（检查上一节输出 data/parsed/道通24年年报/mineru）"
    elif source == "paddleocr_vl":
        # PaddleOCR-VL 输出是按批次子目录拆开的，这里递归读取全部 md 文档块。
        md_files = sorted((BASE_DIR / "paddleocr_vl").glob("*/*.md"))
        assert md_files, "没找到 paddleocr_vl 的 md（检查上一节输出 data/parsed/道通24年年报/paddleocr_vl/**.md）"
    else:
        raise ValueError(f"Unknown source: {source}")

    documents: list[Document] = []
    for md_file in md_files:
        relative_path = md_file.relative_to(BASE_DIR)
        metadata = {
            "source": "道通24年年报",
            "parse_source": source,
            "doc_group": md_file.parent.name,
            "file_name": md_file.name,
            "file_path": relative_path.as_posix(),
            "doc_id": md_file.stem,
        }
        documents.append(
            Document(
                page_content=md_file.read_text(encoding="utf-8"),
                metadata=metadata,
            )
        )

    return documents


raw_docs = load_markdown_documents(SOURCE)
raw_chars = sum(len(doc.page_content) for doc in raw_docs)

print("docs:", len(raw_docs))
print("chars:", raw_chars)
print("sample meta:", raw_docs[0].metadata)
print(raw_docs[0].page_content[:500])


docs: 306
chars: 877063
sample meta: {'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': '道通24年年报_p0001-0050', 'file_name': '道通24年年报_p0001-0050_0.md', 'file_path': 'paddleocr_vl/道通24年年报_p0001-0050/道通24年年报_p0001-0050_0.md', 'doc_id': '道通24年年报_p0001-0050_0'}
# 2024 年度报告

深圳市道通科技股份有限公司

<div style="text-align: center;"><img src="imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>



In [4]:
# 2) Chunk（思路：先按标题切，再做递归切细）

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

child_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", ". ", " ", ""],
    chunk_size=800,
    chunk_overlap=120,
)

chunks = []
for raw_doc in raw_docs:
    header_docs = header_splitter.split_text(raw_doc.page_content)
    if not header_docs:
        header_docs = [Document(page_content=raw_doc.page_content, metadata={})]

    merged_docs = [
        Document(
            page_content=doc.page_content,
            metadata={**raw_doc.metadata, **doc.metadata},
        )
        for doc in header_docs
    ]

    doc_chunks = child_splitter.split_documents(merged_docs)
    for chunk_in_file, doc_chunk in enumerate(doc_chunks):
        doc_chunk.metadata.update(
            {
                "chunk_in_file": chunk_in_file,
            }
        )
        chunks.append(doc_chunk)

for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("chunks:", len(chunks))
print("sample meta:", chunks[0].metadata)
print("sample text:\n", chunks[0].page_content[:300])


chunks: 2069
sample meta: {'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': '道通24年年报_p0001-0050', 'file_name': '道通24年年报_p0001-0050_0.md', 'file_path': 'paddleocr_vl/道通24年年报_p0001-0050/道通24年年报_p0001-0050_0.md', 'doc_id': '道通24年年报_p0001-0050_0', 'h1': '2024 年度报告', 'chunk_in_file': 0, 'chunk_id': 0}
sample text:
 深圳市道通科技股份有限公司  
<div style="text-align: center;"><img src="imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [5]:
# 3) 写入 Chroma（本地持久化）

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL", "text-embedding-3-small")

assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先清空并重建 data/chroma。
# OpenRouter 兼容 OpenAI SDK，这里显式传入 api_key/base_url。
emb = OpenAIEmbeddings(
    model=embed_model,
    api_key=openai_api_key,
    base_url=openai_base_url,
)

vectordb = Chroma(
    collection_name=COLLECTION,
    embedding_function=emb,
    persist_directory=str(CHROMA_DIR),
)

ids = [
    f"{doc.metadata['parse_source']}::{doc.metadata['file_path'].replace('/', '__')}::chunk_{doc.metadata['chunk_in_file']}"
    for doc in chunks
]

existing = vectordb.get(ids=ids, include=[])
existing_ids = existing.get("ids", []) if existing else []
if existing_ids:
    vectordb.delete(ids=existing_ids)

vectordb.add_documents(chunks, ids=ids)
vectordb.persist()

print("persisted:", CHROMA_DIR.resolve())
print("collection:", COLLECTION)
print("embed:", embed_model)
print("env:", ENV_FILE)
print("docs ingested:", len(raw_docs))
print("chunks ingested:", len(chunks))


/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_49128/664063982.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


persisted: /Users/mengbai/Documents/AI-training/RAG_project/data/chroma
collection: autel_annual_report_2024
docs ingested: 306
chunks ingested: 2069


/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_49128/664063982.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
